# 01 · Mapping the 323 Classes to USDA FoodData Central (API) with All Nutrients

Produces a `class → USDA food` table with **all nutrients per 100 g** (plus household portions/gram weights), ready to estimate calories and macros from the predicted class and volume/weight.

**Idea.** The LLM does **not** decide from memory: it only chooses among candidates returned by the USDA API, so it cannot hallucinate IDs. One model (DeepSeek V4 Pro) proposes the mapping; **all** rows are then reviewed manually (second reviewer) and those requiring technical review are flagged.

| Step | What it does |
|---|---|
| 1 | Loads `classes.json` (from `00_datos`) |
| 2 | Model A expands each class into USDA-style English queries (synonyms, raw/cooked/dried state) |
| 3 | API `foods/search` retrieves candidates (Foundation, SR Legacy, FNDDS) |
| 4 | Model A chooses **only among candidates** (validated JSON) |
| 5 | Flags disagreements, `proxy`/`composite`, uncertain state, invalid IDs |
| 6 | `review_queue.csv`: flagged items + audit sample; reviewed manually |
| 7 | Downloads full nutrients for the final IDs → long and wide tables + error rate (Wilson) |

**Before running:** free USDA API key (<https://api.data.gov/signup/>) and OpenCode Go API key, set as secrets/environment variables `USDA_API_KEY` and `OPENCODE_API_KEY`. **Do not paste keys into the notebook or commit them to a repository** (USDA deactivates publicly exposed keys).

**USDA quota:** 1,000 requests/hour per IP; if exceeded, the key is blocked for 1 hour. The notebook uses ~700 calls in total, caches everything, and stops automatically when quota runs low: just rerun later and it resumes where it left off. On Kaggle/Colab the IP is shared with other users, so hitting 429 early and waiting is normal.

In [ ]:
import sys, os, re, json, time, math, random, hashlib, threading, unicodedata, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

try: import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=False); import requests
import numpy as np, pandas as pd
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x

In [ ]:
CFG = dict(
    classes_json = "classes.json",   # next to this notebook; None = autodetect
    # --- USDA API ---
    usda_key_env = "USDA_API_KEY",
    usda_base    = "https://api.nal.usda.gov/fdc/v1",
    data_types   = ["Foundation", "SR Legacy", "Survey (FNDDS)"],   # exclude Branded: per-serving and brand-specific
    page_size = 12,          # results per search
    queries_per_class = 2,   # search queries per class (primary + best alternative)
    n_candidates = 15,       # candidates shown to the LLM
    quota_reserve = 40,      # stop if remaining requests fall below this threshold (X-RateLimit-Remaining)
    # --- LLM (OpenCode Go, OpenAI-compatible endpoint) ---
    api_key_env = "OPENCODE_API_KEY",
    base_url    = "https://opencode.ai/inference/go/openai/v1/chat/completions",
    models_url  = "https://opencode.ai/inference/go/v1/models",
    model_a = "deepseek-v4-pro",     # two distinct families; cell 4 checks that they exist in your plan
    model_b = None,         # single model; the second reviewer is manual review (cell 7)
    temperature = 0, max_tokens = 6000, workers = 3, timeout = 180,
    # --- pipeline ---
    audit_frac = 0.10,       # fraction of auto-accepted items sent to audit
    seed = 42,
    limit = None,            # test with e.g. 15 classes before running all 323
    reviewed_csv = "review_queue_done.csv",
    recipes_csv = "recetas.csv",     # composition of composite dishes (see cell 8)
    out_dir = None,
)
OUT = Path(CFG["out_dir"] or ("/kaggle/working/nutri_map" if Path("/kaggle/working").exists() else "./nutri_map"))
OUT.mkdir(parents=True, exist_ok=True)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])

def load_env_file():
    # Reads the project .env (current folder or a parent) without overwriting existing env vars
    for d in [Path.cwd(), *Path.cwd().parents]:
        f = d / ".env"
        if f.exists():
            for ln in f.read_text(encoding="utf-8").splitlines():
                if "=" in ln and not ln.lstrip().startswith("#"):
                    k, v = ln.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return f
    return None
ENV_FILE = load_env_file()
print("Variables loaded from:", ENV_FILE or "none (using environment variables)")

def get_secret(name):
    k = os.environ.get(name)
    if k: return k
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception: pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception: pass
    raise RuntimeError(f"Missing secret/environment variable {name}.")
print("Output directory:", OUT.resolve())

## 1. Dataset Classes

In [ ]:
def find_classes():
    if CFG["classes_json"]: return Path(CFG["classes_json"])
    c = [Path("classes.json"), Path("food_prep/classes.json"), Path("/kaggle/working/food_prep/classes.json"), Path("/content/food_prep/classes.json")]
    k = Path("/kaggle/input")
    if k.exists(): c += list(k.glob("*/classes.json")) + list(k.glob("*/*/classes.json"))
    for p in c:
        if p.exists(): return p
    raise FileNotFoundError("Cannot find classes.json: set its path in CFG['classes_json'].")

CLS = json.load(open(find_classes(), encoding="utf-8"))
if CFG["limit"]: CLS = CLS[:CFG["limit"]]
def human(s): return re.sub(r"\s+", " ", str(s).replace("-", " ").replace("_", " ")).strip()
print(f"{len(CLS)} classes" + (" (test mode)" if CFG["limit"] else " (expected 323)"))
for c in CLS[:5]: print("  ", c["idx"], c["name"], "| groups:", c.get("old_names", [])[:3])

## 2. Clients: Cache, LLM (OpenCode Go), and USDA API

Everything is saved in `llm_cache.jsonl` and `usda_cache.jsonl`. If you hit the LLM rate limit or USDA quota, just rerun.

In [ ]:
class Cache:
    def __init__(self, path):
        self.path, self.d, self.lock = Path(path), {}, threading.Lock()
        if self.path.exists():
            for ln in open(self.path, encoding="utf-8"):
                try: o = json.loads(ln); self.d[o["k"]] = o["v"]
                except Exception: pass
    def put(self, k, v):
        with self.lock:
            self.d[k] = v
            with open(self.path, "a", encoding="utf-8") as f: f.write(json.dumps({"k": k, "v": v}, ensure_ascii=False) + "\n")
LLM_CACHE, API_CACHE = Cache(OUT / "llm_cache.jsonl"), Cache(OUT / "usda_cache.jsonl")

# ---------------- LLM ----------------
LLM_KEY = get_secret(CFG["api_key_env"])
import uuid
SESSION = str(uuid.uuid4())   # OpenCode Go requires x-opencode-session (one stable session per run)
LHEAD = {"Authorization": f"Bearer {LLM_KEY}", "Content-Type": "application/json",
         "x-opencode-session": SESSION, "User-Agent": "calorie-counter-nutri-map/1.0"}
try:
    ids = [m["id"] for m in requests.get(CFG["models_url"], headers=LHEAD, timeout=30).json().get("data", [])]
    print("Available models in your plan:", ids)
    for m in [x for x in (CFG["model_a"], CFG["model_b"]) if x]:
        if ids and m not in ids: print(f"⚠️ '{m}' not found in list: update CFG['model_a'/'model_b'].")
except Exception as e:
    print("Could not list models:", type(e).__name__, e)

class LLM:
    def __init__(self, model): self.model = model
    def chat(self, system, user):
        body = dict(model=self.model, temperature=CFG["temperature"], max_tokens=CFG["max_tokens"],
                    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
        err = None
        for att in range(6):
            try:
                r = requests.post(CFG["base_url"], headers=LHEAD, json=body, timeout=CFG["timeout"])
                if r.status_code in (429, 500, 502, 503, 504): time.sleep(2 ** att * 2); continue
                if r.status_code == 400 and "temperature" in body and "temperature" in r.text.lower():
                    body.pop("temperature"); continue
                r.raise_for_status()
                return r.json()["choices"][0]["message"].get("content") or ""
            except requests.RequestException as e:
                err = e; time.sleep(2 ** att)
        raise RuntimeError(f"{self.model}: failed after retries ({err})")

def parse_json(txt):
    if not txt: return None
    t = re.sub(r"<think>.*?</think>", "", txt, flags=re.S)
    t = re.sub(r"```(?:json)?", "", t)
    a, b = t.find("{"), t.rfind("}")
    if a < 0 or b <= a: return None
    try: return json.loads(t[a:b + 1])
    except json.JSONDecodeError: return None

def run_stage(llm, stage, items):
    def work(it):
        idx, s, u = it
        key = f"{llm.model}|{stage}|{idx}|{hashlib.md5((s + u).encode()).hexdigest()[:10]}"
        if key in LLM_CACHE.d: return idx, LLM_CACHE.d[key]
        js = None
        for _ in range(3):
            try: js = parse_json(llm.chat(s, u))
            except RuntimeError as e:
                print(f"   ⚠️ {llm.model} · {stage} · class {idx}: {e}"); break
            if js is not None: break
        if js is not None: LLM_CACHE.put(key, js)
        return idx, js
    with ThreadPoolExecutor(CFG["workers"]) as ex:
        res = dict(tqdm(ex.map(work, items), total=len(items), desc=f"{stage} · {llm.model}"))
    print(f"   {stage}/{llm.model}: {sum(v is None for v in res.values())} responses without valid JSON")
    return res
LA = LLM(CFG["model_a"])
LB = LLM(CFG["model_b"]) if CFG["model_b"] else None

# ---------------- USDA FoodData Central ----------------
class QuotaStop(RuntimeError): pass
class FDC:
    def __init__(self, key): self.key, self.low, self.calls = key, False, 0
    def call(self, method, path, body=None):
        if self.low: raise QuotaStop("USDA quota nearly exhausted: rerun this cell in ~1 hour (cache preserves progress).")
        url = CFG["usda_base"] + path
        for att in range(5):
            try:
                r = (requests.post(url, params={"api_key": self.key}, json=body, timeout=60) if method == "POST"
                     else requests.get(url, params={"api_key": self.key}, timeout=60))
            except requests.RequestException:
                time.sleep(2 ** att); continue
            if r.status_code == 429: self.low = True; raise QuotaStop("USDA returned 429 (rate limit reached): wait ~1 hour and rerun.")
            if r.status_code in (500, 502, 503, 504): time.sleep(2 ** att * 2); continue
            r.raise_for_status(); self.calls += 1
            rem = r.headers.get("X-RateLimit-Remaining")
            if rem is not None and rem.isdigit() and int(rem) < CFG["quota_reserve"]: self.low = True
            return r.json()
        raise RuntimeError(f"USDA {path}: failed after retries")
API = FDC(get_secret(CFG["usda_key_env"]))

def parse_nutrients(item):
    # Accepts all three API formats: search (nutrientId/value), abridged (number/amount), and full (nutrient{}/amount)
    out = []
    for n in item.get("foodNutrients", []) or []:
        if isinstance(n.get("nutrient"), dict):
            nu = n["nutrient"]; nid, num, name, unit, val = nu.get("id"), nu.get("number"), nu.get("name"), nu.get("unitName"), n.get("amount")
        else:
            nid, num = n.get("nutrientId"), n.get("nutrientNumber", n.get("number"))
            name, unit, val = n.get("nutrientName", n.get("name")), n.get("unitName"), n.get("value", n.get("amount"))
        if val is None or name is None: continue
        out.append(dict(nutrient_id=nid, number=None if num is None else str(num), name=name, unit=(unit or "").lower(), amount=float(val)))
    return out

def energy_kcal(nuts):
    for num in ("208", "957", "958"):        # energy (kcal); 957/958 = Atwater (Foundation)
        for n in nuts:
            if n["number"] == num and n["unit"] == "kcal": return n["amount"]
    return None

def fdc_search(q):
    key = f"search|{q.lower()}|{'/'.join(CFG['data_types'])}|{CFG['page_size']}"
    if key in API_CACHE.d: return API_CACHE.d[key]
    js = API.call("POST", "/foods/search", {"query": q, "dataType": CFG["data_types"], "pageSize": CFG["page_size"]})
    foods = [dict(fdc_id=str(f["fdcId"]), description=f.get("description", ""), data_type=f.get("dataType", ""),
                  category=f.get("foodCategory", ""), kcal=energy_kcal(parse_nutrients(f))) for f in js.get("foods", [])]
    API_CACHE.put(key, foods); return foods

## 3. Query Expansion (Model A)

In [ ]:
SYS_EXPAND = '''You are a nutritionist expert in food composition tables (USDA FoodData Central).
You receive a class name from a food recognition dataset (English names taken from a Swiss database; they may be generic or very specific).
Return ONLY a JSON object with these keys:
"kind": "ingredient" | "dish" | "product" | "beverage",
"state": "raw" | "cooked" | "dried" | "processed" | "unknown",
"en_query": short USDA-style English query (e.g. "Cabbage, savoy, cooked"),
"alt_en": up to 3 English synonyms or alternatives, from most to least likely (e.g. American names for the same food).'''

def user_expand(c): return f"Class: {human(c['name'])}\nOriginal names grouped into this class: {[human(n) for n in c.get('old_names', [])][:8]}"
EXP = run_stage(LA, "expand", [(c["idx"], SYS_EXPAND, user_expand(c)) for c in CLS])
def exp_of(c):
    e = EXP.get(c["idx"]) or {}
    return dict(kind=e.get("kind", "unknown"), state=e.get("state", "unknown"),
                en=[q for q in [e.get("en_query")] + list(e.get("alt_en") or []) if q] or [human(c["name"])])
for c in CLS[:4]: print(human(c["name"]), "→", exp_of(c))

## 4. Candidates from the USDA API

Per class: primary query + best alternative (+ original name if no results were returned). Results are interleaved and deduplicated. Progress is displayed, and if the quota runs out, the cell stops with a clear message while preserving everything already fetched.

In [ ]:
def candidates_for(c):
    e = exp_of(c); qs = list(dict.fromkeys(e["en"][:CFG["queries_per_class"]] + [human(c["name"])]))
    lists = [fdc_search(q) for q in qs]
    if not any(lists): lists = [fdc_search(human(c["name"]))]
    out, seen = [], set()
    for rank in range(max(map(len, lists), default=0)):
        for L in lists:
            if rank < len(L) and L[rank]["fdc_id"] not in seen:
                seen.add(L[rank]["fdc_id"]); out.append(L[rank])
    return out[:CFG["n_candidates"]]

CAND = {}
try:
    for c in tqdm(CLS, desc="USDA searches"): CAND[c["idx"]] = candidates_for(c)
except QuotaStop as e:
    print("⛔", e, f"| classes with candidates: {len(CAND)}/{len(CLS)}")
    raise
print(f"API calls in this session: {API.calls} (cache avoids repeating previous calls)")
POOL = {f["fdc_id"]: f for L in CAND.values() for f in L}
empty = [c["name"] for c in CLS if not CAND[c["idx"]]]
print(f"Classes with no candidates: {len(empty)}", empty[:8])
def fmt(f): return f"{f['fdc_id']} | {f['description']} | {f['data_type']}" + (f" | {f['kcal']:.0f} kcal/100 g" if f["kcal"] is not None else "")

## 5. Selection (Models A and B) — Candidates Only

If an ID is not in the candidate list, it is discarded and flagged as `id_invalido`.

In [ ]:
SYS_SELECT = '''You are a nutritionist expert in food composition. Match a food class to ONE USDA FoodData Central entry, choosing ONLY from the given candidates. Use the exact candidate id (number only). Never invent ids.
Rules:
- Respect the food state (raw, cooked, dried, fried...). If the class does not specify it, choose the state in which it is normally eaten.
- For simple ingredients prefer Foundation or SR Legacy; for prepared dishes or recipes, Survey (FNDDS).
- "match": "exact" (same food and state) | "close" (same preparation or very similar variety) | "proxy" (different food but nutritionally similar) | "composite" (composite dish or product without a faithful entry; pick the closest or null) | "none" (no candidate fits; id null).
- When in doubt between two valid candidates, choose the more generic one.
- Generic category (fish, meat, cheese, nuts...) with no preparation specified: choose the typical ready-to-eat preparation.
- Branded or regional product with no equivalent: use "proxy" with the closest candidate, or "none".
Reply ONLY with JSON: {"id":"123456|null","match":"...","state_ok":true|false,"reason":"max 25 words"}
"state_ok" is false if the chosen entry does not match the food state.'''

def user_select(c):
    e = exp_of(c)
    return (f"Class: {human(c['name'])}\nGrouped original names: {[human(n) for n in c.get('old_names', [])][:8]}\n"
            f"Estimated type: {e['kind']} | estimated state: {e['state']}\n\nUSDA CANDIDATES:\n" + ("\n".join(fmt(f) for f in CAND[c["idx"]]) or "(none)"))

def norm_sel(js, c):
    out = dict(id=None, match="none", invalid=False, state_ok=True, reason="", failed=js is None)
    if js is None: out.update(state_ok=False, reason="no valid response"); return out
    i = js.get("id"); m = str(js.get("match", "none")).lower()
    out["state_ok"] = bool(js.get("state_ok", True)); out["reason"] = str(js.get("reason", ""))[:200]
    i = None if i is None else re.sub(r"^[Uu]:", "", str(i).strip())
    if i in (None, "", "null", "None"): i = None
    elif i not in {f["fdc_id"] for f in CAND[c["idx"]]}: out["invalid"] = True; i = None; m = "none"
    m = m if m in ("exact", "close", "proxy", "composite", "none") else "none"
    if i is None and m != "composite": m = "none"       # 'composite' without id is preserved: the dish needs a decision (recipe/proxy)
    out["id"], out["match"] = i, m
    return out

SEL = {}
items = [(c["idx"], SYS_SELECT, user_select(c)) for c in CLS]
for tag, llm in [t for t in (("A", LA), ("B", LB)) if t[1]]:
    raw = run_stage(llm, "select", items)
    SEL[tag] = {c["idx"]: norm_sel(raw[c["idx"]], c) for c in CLS}

## 6. Combine, Flag, and Decide What Needs Review

Statuses: **`auto_ok`** (both agree and match is `exact`/`close`, no flags), **`no_match`** (both agree there is no entry), and **`review`** (any flag: `desacuerdo`, `proxy`, `composite`, `estado`, `id_invalido`, `sin_respuesta`, `contradiccion` — says `none` yet still picks an ID).

In [ ]:
RANK = {"exact": 0, "close": 1, "proxy": 2, "composite": 3, "none": 4}
def dsc(i): return POOL[i]["description"] if i in POOL else ""
REVISAR_TODO = len(SEL) == 1          # with a single model, nothing is accepted without human review

def combine(c):
    i = c["idx"]; sels = [SEL[t][i] for t in SEL]; a = sels[0]
    b = SEL["B"][i] if "B" in SEL else dict(id=None, match="none", invalid=False, state_ok=True, reason="", failed=False)
    flags = []
    agree = len({s["id"] for s in sels}) == 1
    worst = max((s["match"] for s in sels), key=lambda m: RANK[m])
    if not agree: flags.append("desacuerdo")
    elif worst in ("proxy", "composite"): flags.append(worst)
    if any(s["invalid"] for s in sels): flags.append("id_invalido")
    if not all(s["state_ok"] for s in sels): flags.append("estado")
    if any(s["failed"] for s in sels): flags.append("sin_respuesta")
    if any(s["match"] == "none" and s["id"] for s in sels): flags.append("contradiccion")
    draft = (a["id"] or "") if agree else ""
    st = "review" if (flags or REVISAR_TODO) else ("no_match" if not draft else "auto_ok")
    return dict(idx=i, clase=c["name"], agrupa=" | ".join(c.get("old_names", [])[:6]), status=st, flags=";".join(sorted(set(flags))),
                usda_draft=draft, usda_draft_desc=dsc(draft), usda_kcal=POOL[draft]["kcal"] if draft in POOL else "",
                usda_A=a["id"] or "", usda_A_desc=dsc(a["id"] or ""), usda_B=b["id"] or "", usda_B_desc=dsc(b["id"] or ""),
                usda_match=worst if agree else "disputed", razon_A=a["reason"], razon_B=b["reason"])

DF = pd.DataFrame([combine(c) for c in CLS])
print(DF["status"].value_counts().to_string())
print("\nMost frequent flags:", Counter(f for s in DF["flags"] for f in s.split(";") if f).most_common(8))

## 7. Review Queue

`review_queue.csv` contains all flagged rows plus a **random audit sample** (`audit`) of auto-accepted rows. Open it in Excel/Sheets, fill in `decision`, and save it as `review_queue_done.csv`:

- **`ok`** → accept the draft (`usda_draft`).
- **`cambiar`** → enter the correct FDC id in `final_usda` (can be A's or B's id from the row, or any id from <https://fdc.nal.usda.gov>).
- **`sin_match`** → the class has no valid entry.
- **`receta`** → composite dish: computed from the rows in `recetas.csv` (cell 8).

For `review` rows with a disagreement, the draft is blank: you will usually use `cambiar` with A's or B's id (their descriptions are in the row).

In [ ]:
rng = np.random.RandomState(CFG["seed"])
passed = DF.index[DF["status"].isin(["auto_ok", "no_match"])].to_numpy()
n_aud = int(math.ceil(CFG["audit_frac"] * len(passed))) if len(passed) else 0
aud = set(rng.choice(passed, n_aud, replace=False).tolist()) if n_aud else set()
DF["status_final"] = [("audit" if i in aud else s) for i, s in zip(DF.index, DF["status"])]

cols = ["idx", "clase", "agrupa", "status_final", "flags", "usda_draft", "usda_draft_desc", "usda_kcal", "usda_match",
        "usda_A", "usda_A_desc", "usda_B", "usda_B_desc", "razon_A", "razon_B"]
Q = DF[DF["status_final"].isin(["review", "audit"])][cols].copy()
Q["decision"] = ""; Q["final_usda"] = ""; Q["nota"] = ""
Q = Q.sort_values(["status_final", "idx"], ascending=[False, True])   # review first, then audit
Q.to_csv(OUT / "review_queue.csv", index=False, encoding="utf-8-sig")
DF.to_csv(OUT / "mapping_draft.csv", index=False, encoding="utf-8-sig")
print(DF["status_final"].value_counts().to_string())
print(f"\n→ {len(Q)} rows for manual review out of {len(DF)} classes: {OUT / 'review_queue.csv'}")

## 8. Final Table with All Nutrients

After reviewing, save `review_queue_done.csv` and run this cell. It downloads the **full** nutrient profiles for the final IDs (`POST /foods`, 20 per call, ~17 calls) and generates:

- `nutrition_wide.csv`: one row per class, one column per nutrient (per 100 g). Blank = that nutrient was not measured for that entry.
- `nutrition_long.csv`: long format (class, fdc_id, nutrient, unit, amount).
- `nutrient_dictionary.csv`: id, number, name, and unit of each nutrient.
- `nutrition_map.json`: per class, IDs, nutrients, and household portions (`gramWeight`), useful for converting volume/measures to grams.

It also estimates the **error rate of unreviewed items** from the audit sample (95% Wilson interval).

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - h) / d, (c + h) / d)

def fetch_details(ids):
    ids = sorted({str(i) for i in ids if i}); got = {}
    for k in range(0, len(ids), 20):
        chunk = ids[k:k + 20]; key = "foods|full|" + ",".join(chunk)
        if key not in API_CACHE.d:
            js = API.call("POST", "/foods", {"fdcIds": [int(i) for i in chunk], "format": "full"})
            API_CACHE.put(key, js if isinstance(js, list) else js.get("foods", []))
        for f in API_CACHE.d[key]: got[str(f["fdcId"])] = f
    return got

def load_recipes():
    # recetas.csv: one row per ingredient; grams = grams of that ingredient (as consumed) per 100 g of finished dish
    p = Path(CFG["recipes_csv"])
    if not p.exists(): return {}
    R = pd.read_csv(p, dtype=str).fillna("")
    out = {}
    for idx, g in R.groupby("idx"):
        ings = [dict(nombre=r["ingrediente"], fdc_id=re.sub(r"\D", "", r["fdc_id"]), g=float(r["gramos_100g_plato"] or 0)) for _, r in g.iterrows()]
        total = sum(i["g"] for i in ings)
        if abs(total - 100) > 5: print(f"⚠️ recipe {idx}: ingredients sum to {total:.0f} g (should sum to ~100 g per 100 g of finished dish)")
        out[int(idx)] = dict(ings=ings, porcion_g=next((float(x) for x in g["porcion_g"] if x), None))
    return out

def _energia_208(nuts):
    # Foundation entries only carry Atwater energy (957/958): map it to 208 so the sum does not count it as 0
    if any(n["number"] == "208" for n in nuts): return nuts
    for num in ("957", "958"):
        for n in nuts:
            if n["number"] == num and n["unit"] == "kcal":
                return nuts + [dict(n, number="208", nutrient_id=1008, name="Energy")]
    return nuts

def recipe_profile(rec, det):
    # weighted sum by grams. If an ingredient does not report a nutrient (e.g. sugars in an oil), it counts as 0
    per = []
    for ing in rec["ings"]:
        f = det.get(ing["fdc_id"]) if ing["fdc_id"] else None
        if f is None: raise ValueError(f"ingredient '{ing['nombre']}' has no valid id ({ing['fdc_id'] or 'empty'})")
        per.append({(n["nutrient_id"], n["number"]): n for n in _energia_208(parse_nutrients(f))})
    allk = set().union(*map(set, per)) if per else set()
    faltan = sum(1 for p in per for k in allk if k not in p)
    if faltan: print(f"   ℹ️ recipe: {faltan} nutrient value(s) missing in some ingredient, counted as 0")
    out = []
    for k in sorted(allk, key=lambda k: str(k[1])):
        amt = sum((p[k]["amount"] if k in p else 0) * ing["g"] / 100 for p, ing in zip(per, rec["ings"]))
        base = next(p[k] for p in per if k in p)
        out.append(dict(base, amount=round(amt, 4)))
    return out

def finalize(path=None):
    path = Path(path or OUT / CFG["reviewed_csv"])
    rev = pd.read_csv(path, dtype=str).fillna("") if path.exists() else pd.DataFrame(columns=["idx", "decision"])
    if not path.exists(): print(f"⚠️ Cannot find {path}: using draft without human review.") if not path.exists() else print(f"Decisions read from: {path}")
    R = {int(r["idx"]): r for _, r in rev.iterrows()}
    REC = load_recipes()
    rows, problemas, pendientes, aud_n, aud_err = [], [], 0, 0, 0
    for _, d in DF.iterrows():
        i = int(d["idx"]); r = R.get(i); dec = r["decision"].strip().lower() if r is not None else ""
        fid, match = d["usda_draft"], d["usda_match"]
        if dec not in ("", "ok", "cambiar", "sin_match", "receta"):
            problemas.append(f"{i} {d['clase']}: unknown decision '{dec}'")
        elif dec == "cambiar":
            fid, match = re.sub(r"\D", "", r["final_usda"]), "manual"
            if not fid: problemas.append(f"{i} {d['clase']}: 'cambiar' without final_usda")
        elif dec == "sin_match": fid, match = "", "none"
        elif dec == "receta":
            fid, match = "", "receta"
            if i not in REC: problemas.append(f"{i} {d['clase']}: 'receta' but no rows found for this class in {CFG['recipes_csv']}")
        elif dec == "ok" and not fid and d["status_final"] != "no_match":
            problemas.append(f"{i} {d['clase']}: 'ok' on an empty draft (use 'cambiar', 'receta', or 'sin_match')")
        elif not dec and d["status_final"] == "review": pendientes += 1
        if d["status_final"] == "audit" and dec in ("ok", "cambiar", "sin_match", "receta"):
            aud_n += 1; aud_err += (fid != d["usda_draft"]) or dec == "receta"
        rows.append(dict(idx=i, clase=d["clase"], status=d["status_final"], reviewed=bool(dec), flags=d["flags"], fdc_id=fid, match=match))
    if problemas:
        print(f"⚠️ {len(problemas)} incomplete or invalid decisions (those rows remain without an id):")
        for p in problemas[:30]: print("   -", p)
    M = pd.DataFrame(rows)
    used = {r["idx"]: REC[r["idx"]] for r in rows if r["match"] == "receta" and r["idx"] in REC}
    det = fetch_details(list(M["fdc_id"]) + [g["fdc_id"] for rec in used.values() for g in rec["ings"]])
    long_rows, wide_rows, js_out, nd = [], [], [], {}
    for _, m in M.iterrows():
        i = int(m["idx"])
        if i in used:
            rec = used[i]
            try: nuts = recipe_profile(rec, det)
            except ValueError as e: print(f"⚠️ recipe {i} ({m['clase']}): {e}"); nuts = []
            fid_lab, extra = "RECETA", dict(ingredientes=rec["ings"])
            info = dict(fdc_description=f"RECETA ({len(rec['ings'])} ingredients)", fdc_data_type="receta", kcal_100g=energy_kcal(nuts))
            portions = [dict(amount=1, gram_weight=rec["porcion_g"], modifier=None, description="typical dish serving", unit="g")] if rec["porcion_g"] else []
        else:
            f = det.get(m["fdc_id"]) if m["fdc_id"] else None
            if m["fdc_id"] and f is None: print(f"⚠️ the API did not return id {m['fdc_id']} (class {m['clase']})")
            nuts = parse_nutrients(f) if f else []
            fid_lab, extra = m["fdc_id"], {}
            info = dict(fdc_description=(f or {}).get("description", ""), fdc_data_type=(f or {}).get("dataType", ""), kcal_100g=energy_kcal(nuts))
            portions = [dict(amount=p.get("amount"), gram_weight=p.get("gramWeight"), modifier=p.get("modifier"), description=p.get("portionDescription"),
                             unit=(p.get("measureUnit") or {}).get("name")) for p in (f or {}).get("foodPortions", [])]
        for n in nuts:
            nd[(n["nutrient_id"], n["number"])] = (n["name"], n["unit"])
            long_rows.append(dict(idx=i, clase=m["clase"], fdc_id=fid_lab, **n))
        js_out.append(dict(m.to_dict(), **info, nutrients={f"{n['name']} ({n['unit']})": n["amount"] for n in nuts}, portions=portions, **extra))
        wide_rows.append((m.to_dict(), info, nuts))
    L = pd.DataFrame(long_rows)
    # wide-format columns: "Name (unit)"; if two nutrients collide, append their number
    lab = {}
    for (nid, num), (name, unit) in nd.items(): lab[(nid, num)] = f"{name} ({unit})"
    cnt = Counter(lab.values())
    lab = {k: (v if cnt[v] == 1 else f"{v} [{k[1]}]") for k, v in lab.items()}
    W = []
    for m, info, nuts in wide_rows:
        row = {**m, **info}; row.update({lab[(n["nutrient_id"], n["number"])]: n["amount"] for n in nuts}); W.append(row)
    W = pd.DataFrame(W)
    W.to_csv(OUT / "nutrition_wide.csv", index=False, encoding="utf-8-sig")
    L.to_csv(OUT / "nutrition_long.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame([dict(nutrient_id=k[0], number=k[1], name=v[0], unit=v[1], column=lab[k]) for k, v in nd.items()],
                 columns=["nutrient_id", "number", "name", "unit", "column"]).sort_values("number").to_csv(OUT / "nutrient_dictionary.csv", index=False)
    json.dump(js_out, open(OUT / "nutrition_map.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"Classes: {len(M)} | with USDA entry: {(M['fdc_id'] != '').sum()} | recipes: {len(used)} | distinct nutrients: {len(nd)} | undecided 'review' rows: {pendientes}" + ("  ⚠️ review them before using the mapping" if pendientes else ""))
    if len(W): print("Data types used:", W["fdc_data_type"].value_counts().to_dict())
    if aud_n:
        lo, hi = wilson(aud_err, aud_n)
        print(f"Audit: {aud_err} differences in {aud_n} rows → rate {aud_err/aud_n:.1%} (95% CI {lo:.1%}–{hi:.1%}) for items not manually reviewed")
    else: print("No reviewed audit rows yet: cannot estimate error rate.")
    return W
FIN = finalize()

### Notes for Reporting and Downstream Use
- **Reproducibility:** the API returns the current version of each database. The final tables already contain the values used, and `usda_cache.jsonl` stores the raw responses; archive them along with the execution date. Also save `CFG` (models, `temperature=0`, data types, audit seed).
- **Per 100 g:** Foundation, SR Legacy, and FNDDS values are expressed per 100 g of edible portion; `Branded` was excluded intentionally (reported per serving and with variable quality).
- **Uneven nutrient coverage:** each data type measures a different set of nutrients (FNDDS has fewer nutrients than Foundation/SR Legacy), so some columns will be blank. `fdc_data_type` is kept in the table for reference.
- **Composite dishes (`match` = `receta`):** computed by summing the nutrients from `recetas.csv` (grams per 100 g of finished dish, each ingredient with a USDA id). Only nutrients present in all ingredients are kept.
- If you add more classes or change the prompt, the cache is invalidated only for what changed (the key includes a hash of the prompt).